Remember, Pooling Layers aim to reduce the spatial dimensions of the feature maps. In ML, we aim to reduce computational load by balancing the number of paramters we want to use for a given model. Pooling layers summarize the presence of features in patches of the feature map, pooling. We'll go over a few common pooling techniques. 
First, with a feature map with $n_h \times n_w \times n_c $, the dimensions of the output after a pooling layer assuming valid pooling will be:
$$
\left( \frac{n_h - f + 1}{s} \right) \times \left( \frac{n_w - f + 1}{s} \right) \times n_c
$$
If we are doing "same" pooling, we'll use the formula: 
$$
\left( \frac{n_h }{s} \right) \times \left( \frac{n_w}{s} \right) \times n_c
$$
Where:  
* $n_h \rightarrow $ height of the feature map  
* $n_w \rightarrow $ width of the feature map  
* $n_c \rightarrow $ number of channels in the feature map  
* $f \rightarrow $ size of the pooling filter  
* $s \rightarrow $ stride length  



**Max Pooling** For this implementation, I will be incorporating the Max Pooling into the class. The formula that I will be using is as follows:
$$
P_{\text{max}}(i, j) = \max_{0 \le a < n, \, 0 \le b < n} F(i \cdot s + a, \, j \cdot s + b)
$$
Where:
* $P_{max} (i, j)\text{: the pooled value at position } (i,j)$
* $F$: the input feature map
* $n$: the pooling windows size (e.g, 2x2, 3x3, etc)
* $s$: the stride
* $a, b$: iterate over the window dimensions

In this case, lets imagine that after we recieve our output tensor from Layer_Conv, we'll have output = (number of samples, output height, output width, number of channels). For now, lets assume we performed a convolution on the sample batch with size 128, with the same 28x28 pixel grid we have been working on, and we passed in 2 filters for the model. The result will be 
output = (128, 28, 28, 2)
If we wanted to do max pooling, we could say that we would like to take the max of a 2x2 block. Conceputally, we'd 
*  Store the filter size (e.g (2, 2))
*  Store the stride (e.g (2, 2))
*  Store the pooling type, in this case type = "max"  

We could think of this as:  
“Remember how big the pooling window should be and how far to move it each step.” 

Now, for the forward pass, we'd like to know how big the filter will be, how big the stride will be for each iteration, the type of padding used if any (usually none), and finally the pooling type (we'll use only max for this implementation). 
 


In [ ]:
import numpy as np
from numpy.lib.stride_tricks import as_strided
class Pooling: 
    def __init__(self, filter_size = (2, 2), strides = (2, 2),
                  padding = "valid", pooling_type = "max"):
        self.filter_size = filter_size
        self.strides = strides
        self.padding = padding
        self.pooling_type = pooling_type

    def forward(self, inputs):
        #Inputs should be of shape (S, H_in, W_in, C = D_in) 
        if inputs.ndim != 4:
            raise ValueError(f"Expected a 4D tensor, got {inputs.ndim} instead.")
        S, H_in, W_in, C = inputs.shape
        fH, fW = self.filter_size
        sH, sW = self.strides

        padding = self.padding
        if padding == "valid":
            H_out = np.floor((H_in - fH) / sH) + 1
            W_out = np.floor((W_in - fW) / sW) + 1
        
        elif padding == "same":
            pad_h = max((H_out - 1) * sH + fH - H_in, 0)
            pad_w = max((W_out - 1) * sW + fW - W_in, 0)
            pad_top = pad_h // 2
            pad_bottom = pad_h - pad_top
            pad_left = pad_w // 2
            pad_right = pad_w - pad_left
            inputs = np.pad(inputs, ((0,0), (pad_top,pad_bottom), (pad_left,pad_right), (0,0)), mode='constant')
        else: 
            raise ValueError(f"Expected padding == valid or same, recieved {padding} instead")

        #cast our output dimensions into ints from floats. 
        H_out, W_out = int(H_out), int(W_out)

        #create output tensor with new sizes
        self.output = np.zeros(shape = (S, H_out, W_out, C))
        self.inputs = inputs
        patches = as_strided(
            inputs,
            shape = (S, H_out, W_out, fH, fW, C), 
            strides = (
                inputs.strides[0],      #step between samples
                inputs.strides[1] * sH, #step between rows
                inputs.strides[2] * sW, #step between columns
                inputs.strides[1],      #Move down 1 row inside patch
                inputs.strides[2],      #move right 1col inside patch
                inputs.strides[3],      #step between each channel
            )
        )

        if self.pooling_type == "max":
            pooled = patches.max(axis = (3, 4)) 
        
        #Store both of these for backprop
        self.inputs = inputs
        self.output = pooled
        return self.output

